In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -U gdown

  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.1
    Uninstalling gdown-5.2.1:
      Successfully uninstalled gdown-5.2.1


In [ ]:
!gdown "https://drive.google.com/uc?id=1ImYUSLk9JbgHXOemfvyiDiirluZHPeQw" -O medqa.zip

Downloading...
From (original): https://drive.google.com/uc?id=1ImYUSLk9JbgHXOemfvyiDiirluZHPeQw
From (redirected): https://drive.google.com/uc?id=1ImYUSLk9JbgHXOemfvyiDiirluZHPeQw&confirm=t&uuid=67fa2b55-0902-41cd-8cc1-700c9389df62
To: /content/medqa.zip
100% 132M/132M [00:05<00:00, 24.0MB/s]


In [ ]:
!unzip -q medqa.zip -d medqa_data

In [ ]:
import os

for root, dirs, files in os.walk("medqa_data"):
    print(root)
    for f in files[:3]:
        print("  ", f)

medqa_data
medqa_data/data_clean
   .DS_Store
medqa_data/data_clean/questions
   .DS_Store
medqa_data/data_clean/questions/Taiwan
   .DS_Store
   train.jsonl
   test.jsonl
medqa_data/data_clean/questions/Taiwan/tw_translated_jsonl
   .DS_Store
medqa_data/data_clean/questions/Taiwan/tw_translated_jsonl/zh
   dev-2zh.jsonl
   train-2zh.jsonl
   test-2zh.jsonl
medqa_data/data_clean/questions/Taiwan/tw_translated_jsonl/en
   train-2en.jsonl
   dev-2en.jsonl
   test-2en.jsonl
medqa_data/data_clean/questions/Taiwan/metamap
   .DS_Store
medqa_data/data_clean/questions/Taiwan/metamap/train
   .DS_Store
   tw_train.jsonl
medqa_data/data_clean/questions/Taiwan/metamap/test
   .DS_Store
   tw_test.jsonl
medqa_data/data_clean/questions/Taiwan/metamap/dev
   .DS_Store
   tw_dev.jsonl
medqa_data/data_clean/questions/US
   .DS_Store
   train.jsonl
   US_qbank.jsonl
medqa_data/data_clean/questions/US/4_options
   phrases_no_exclude_test.jsonl
   phrases_no_exclude_train.jsonl
   phrases_no_exclude_dev

In [ ]:
#Imports & Config
import json
import random
from pathlib import Path
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)

DATA_DIR = Path("medqa_data/data_clean/questions/US")
OUT_DIR  = Path("processed")
OUT_DIR.mkdir(exist_ok=True)

# System prompt
SYSTEM_PROMPT = (
    "You are a medical AI assistant trained on clinical questions. "
    "Answer clearly, accurately, and concisely."
)

# % of MCQ-style samples
MCQ_RATIO = 0.15

In [ ]:
def build_chat(row: dict) -> dict:
    question = row.get("question", "").strip()
    options  = row.get("options", {})

    answer_key = row.get("answer_idx", "")
    answer_key = answer_key.strip()[:1].upper()

    if answer_key and answer_key in options:
        answer_text = options[answer_key]
    else:
        answer_text = row.get("answer", "").strip()

    if not question or not answer_text:
        return None

    use_mcq = random.random() < MCQ_RATIO

    if use_mcq and options:
        options_text = "\n".join(
            f"{k}) {v}" for k, v in sorted(options.items())
        )
        user_part = f"{question}\n\nOptions:\n{options_text}"
        assistant_part = (
            f"The correct answer is {answer_key}) {answer_text}."
            if answer_key else
            f"The correct answer is {answer_text}."
        )
    else:
        user_part      = question
        assistant_part = f"The answer is {answer_text}."

    text = (
        f"<|system|>\n{SYSTEM_PROMPT}\n\n"
        f"<|user|>\n{user_part}\n\n"
        f"<|assistant|>\n{assistant_part}"
    )
    return {"text": text}

In [ ]:
def process_file(src: Path, dst: Path):
    examples = []

    with open(src, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            try:
                row = json.loads(line)
            except:
                continue

            ex = build_chat(row)
            if ex:
                examples.append(ex)

    random.shuffle(examples)

    with open(dst, "w", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

    print(f"✓ {dst.name} → {len(examples):,} samples")

    return examples

In [ ]:
# Step 1: Load and process full train set
train_src = DATA_DIR / "train.jsonl"
train_dst = OUT_DIR / "train_full.jsonl"

train_examples = process_file(train_src, train_dst)


split_idx = int(0.9 * len(train_examples))

train_split = train_examples[:split_idx]
val_split   = train_examples[split_idx:]


# Step 3: Save splits
import json

with open(OUT_DIR / "train.jsonl", "w", encoding="utf-8") as f:
    for ex in train_split:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

with open(OUT_DIR / "val.jsonl", "w", encoding="utf-8") as f:
    for ex in val_split:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"✓ train.jsonl → {len(train_split):,} samples")
print(f"✓ val.jsonl   → {len(val_split):,} samples")


# Step 4: Process test set
test_src = DATA_DIR / "test.jsonl"
test_dst = OUT_DIR / "test.jsonl"

process_file(test_src, test_dst)

print("\nProcessing complete")

✓ train_full.jsonl → 10,178 samples
✓ train.jsonl → 9,160 samples
✓ val.jsonl   → 1,018 samples
✓ test.jsonl → 1,273 samples

Processing complete


In [ ]:
with open("processed/train.jsonl") as f:
    import json
    ex = json.loads(f.readline())
    print(ex["text"])

<|system|>
You are a medical AI assistant trained on clinical questions. Answer clearly, accurately, and concisely.

<|user|>
A 5-year-old girl presents with a rash and a persistent fever of 41.0°C (105.8°F), not relieved by Tylenol. The patient’s mother says that her symptoms started 5 days ago and have not improved. The rash started on her trunk and now is present everywhere including the palms and soles. Her birth history is normal. Her pulse is 120/min and respiratory rate is 22/min. On physical examination, the patient is agitated and ill-appearing. There is significant swelling of the distal upper and lower extremities bilaterally. The pharynx is hyperemic (see image). Generalized edema with non-palpable cervical lymphadenopathy is noted. Muscle tone is normal. Remainder of exam is unremarkable. Laboratory findings are significant for the following:
Laboratory test
Hb 9 g/dL
RBC 3.3/mm3
Neutrophilic leukocytosis 28,000/mm3
Normal platelet count 200,000/mm3
Serum ɣ-GT increased
Hy

In [ ]:
!ls processed

test.jsonl  train_full.jsonl  train.jsonl  val.jsonl


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "processed/train.jsonl",
        "validation": "processed/val.jsonl"
    }
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 9160
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1018
    })
})


In [ ]:
print(dataset["train"][0]["text"])

<|system|>
You are a medical AI assistant trained on clinical questions. Answer clearly, accurately, and concisely.

<|user|>
A 5-year-old girl presents with a rash and a persistent fever of 41.0°C (105.8°F), not relieved by Tylenol. The patient’s mother says that her symptoms started 5 days ago and have not improved. The rash started on her trunk and now is present everywhere including the palms and soles. Her birth history is normal. Her pulse is 120/min and respiratory rate is 22/min. On physical examination, the patient is agitated and ill-appearing. There is significant swelling of the distal upper and lower extremities bilaterally. The pharynx is hyperemic (see image). Generalized edema with non-palpable cervical lymphadenopathy is noted. Muscle tone is normal. Remainder of exam is unremarkable. Laboratory findings are significant for the following:
Laboratory test
Hb 9 g/dL
RBC 3.3/mm3
Neutrophilic leukocytosis 28,000/mm3
Normal platelet count 200,000/mm3
Serum ɣ-GT increased
Hy

In [ ]:
import shutil

# Create a folder in Drive to store the processed data

DRIVE_OUT = "/content/drive/MyDrive/medqa_processed"

import os
os.makedirs(DRIVE_OUT, exist_ok=True)

# Copy all processed files to Drive
for f in ["train.jsonl", "val.jsonl", "test.jsonl", "train_full.jsonl"]:
    src = f"processed/{f}"
    dst = f"{DRIVE_OUT}/{f}"
    shutil.copy(src, dst)
    print(f"✅ Saved {f} → {dst}")

print("\n All files saved to Google Drive!")

✅ Saved train.jsonl → /content/drive/MyDrive/medqa_processed/train.jsonl
✅ Saved val.jsonl → /content/drive/MyDrive/medqa_processed/val.jsonl
✅ Saved test.jsonl → /content/drive/MyDrive/medqa_processed/test.jsonl
✅ Saved train_full.jsonl → /content/drive/MyDrive/medqa_processed/train_full.jsonl

🎉 All files saved to Google Drive!
